# 🐼 Panda AI — One-Click Deploy
Deploy the full Panda AI gateway (API + Dashboard) on Colab.
**Runtime → Run all (Ctrl+F9)**

## 1️⃣ Install System

In [ ]:
# Node 20 (Colab apt gives Node 12, too old for Next.js)
!curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1

# cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf && mv /tmp/cf /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import subprocess, sys
v = subprocess.check_output(['node', '--version']).decode().strip()
assert v.startswith('v20'), f'Need Node 20, got {v}'
print(f'✅ Node {v} | Python {sys.version.split()[0]}')

## 2️⃣ Clone + Python deps

In [ ]:
import os, subprocess

os.chdir('/')
!rm -rf /content/Panda-Ai
!git clone -q https://github.com/ferelking242/Panda-Ai.git /content/Panda-Ai
os.chdir('/content/Panda-Ai')

# Clean lockfiles that confuse Next.js
!rm -f bun.lock bun.lockb

# Python dependencies (from the cloned repo)
!pip install -q -r requirements.txt --root-user-action=ignore 2>&1 | grep -v WARNING | tail -3
!patchright install chromium 2>&1 | tail -1

# .env
with open('.env', 'w') as f:
    f.write('PROVIDER=chatgpt\nHEADLESS=true\nAPI_HOST=0.0.0.0\nAPI_PORT=8000\nPOOL_SIZE=1\nLOG_LEVEL=INFO\n')

print('✅ Repo cloned + Python deps installed')

## 3️⃣ Build Dashboard

In [ ]:
import os, subprocess

os.chdir('/content/Panda-Ai/dashboard')

if os.path.exists('.next/BUILD_ID'):
    bid = open('.next/BUILD_ID').read().strip()
    print(f'✅ Dashboard already built (BUILD_ID: {bid}) — skipping')
else:
    !npm install --no-audit --no-fund 2>&1 | tail -2
    print('🔨 Building dashboard...')
    result = subprocess.run(
        ['npm', 'run', 'build'],
        cwd='/content/Panda-Ai/dashboard',
        capture_output=True, text=True, timeout=180
    )
    if result.returncode != 0:
        print(f'❌ Build FAILED (exit {result.returncode})')
        print(result.stderr[-2000:] if result.stderr else '')
        raise RuntimeError('Dashboard build failed')
    assert os.path.exists('.next/BUILD_ID'), 'BUILD_ID missing after build'
    bid = open('.next/BUILD_ID').read().strip()
    print(f'✅ Dashboard built (BUILD_ID: {bid})')

os.chdir('/content/Panda-Ai')

## 4️⃣ Start Services

In [ ]:
import subprocess, time, os, sys

# Kill any leftover processes
!fuser -k 8000/tcp 2>/dev/null || true
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(2)

env_base = {**os.environ, 'PYTHONUNBUFFERED': '1'}

# Start API server
api_log = open('/tmp/api.log', 'w')
api_proc = subprocess.Popen(
    [sys.executable, '-m', 'src.api.server'],
    cwd='/content/Panda-Ai', env=env_base,
    stdout=api_log, stderr=subprocess.STDOUT
)
print('🚀 API starting...')

# Start Dashboard
dash_log = open('/tmp/dash.log', 'w')
dash_env = {**env_base, 'PORT': '5000', 'API_ORIGIN': 'http://127.0.0.1:8000', 'NODE_ENV': 'production'}
dash_proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='/content/Panda-Ai/dashboard', env=dash_env,
    stdout=dash_log, stderr=subprocess.STDOUT
)
print('📊 Dashboard starting...')

# Wait for API
import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3)
        print('✅ API healthy')
        break
    except: pass
else:
    print('⚠️ API not responding')

# Wait for Dashboard
for i in range(15):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://127.0.0.1:5000', timeout=3)
        print('✅ Dashboard healthy')
        break
    except: pass
else:
    print('⚠️ Dashboard not responding — check /tmp/dash.log')

## 5️⃣ Public URLs + Token

In [ ]:
import json, urllib.request, re, threading, subprocess, time

# Generate token from the running API
try:
    req = urllib.request.Request(
        'http://127.0.0.1:8000/api/dashboard/token/generate',
        data=json.dumps({'name': 'colab', 'scope': ['*']}).encode(),
        headers={'Content-Type': 'application/json'}, method='POST'
    )
    api_token = json.loads(urllib.request.urlopen(req).read())['token']
except Exception as e:
    print(f'⚠️ Token generation failed: {e}')
    import secrets; api_token = 'pnd_' + secrets.token_hex(16)

# Start cloudflared tunnels
urls = {}
def tunnel(port, name):
    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            urls[name] = m.group(0)
            print(f'  🔗 {name}: {m.group(0)}')
            break

threading.Thread(target=tunnel, args=(8000, 'API'), daemon=True).start()
threading.Thread(target=tunnel, args=(5000, 'Dashboard'), daemon=True).start()

for _ in range(30):
    time.sleep(1)
    if len(urls) >= 2:
        break

print()
print('═══════════════════════════════════════════')
print('  🐼 PANDA AI DEPLOYED')
print('═══════════════════════════════════════════')
if 'API' in urls: print(f'  🤖 API:      {urls["API"]}/v1')
if 'Dashboard' in urls: print(f'  📊 Dashboard: {urls["Dashboard"]}')
print(f'  🔑 Token: {api_token}')
print('═══════════════════════════════════════════')

## 📋 Quick Test

In [ ]:
import urllib.request, json
base = 'http://127.0.0.1:8000'
health = json.loads(urllib.request.urlopen(f'{base}/healthz').read())
print(f'✅ Health: {health}')
req = urllib.request.Request(f'{base}/v1/models', headers={'Authorization': f'Bearer {api_token}'})
models = json.loads(urllib.request.urlopen(req).read())
print(f'✅ Models: {[m["id"] for m in models["data"][:5]]}')
print(f'\nbase_url="{urls.get("API","?")}/v1"  api_key="{api_token}"')

## 🔧 Debug

In [ ]:
import subprocess as sp
print('=== API LOG ===')
sp.run(['tail', '-15', '/tmp/api.log'])
print('\n=== DASHBOARD LOG ===')
sp.run(['tail', '-15', '/tmp/dash.log'])
print('\n=== NODE ===')
sp.run(['node', '--version'])